# Daum Movie Review Notebook

`daum_movie_review.csv`를 읽어서 감정분류용 데이터로 바꾸고, `preprocessing.py`와 `model.py`를 연결해 실행하는 노트북이다.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from preprocessing import DEFAULT_STOPWORDS, KoreanSentimentDataset, preprocess_dataframe
from model import BiLSTMSentimentClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
df.head()

In [ ]:
print(df.columns)

review_col = 'review'
rating_col = 'rating'

data = df[[review_col, rating_col]].copy()
data.columns = ['review', 'rating']
data['sentiment'] = (data['rating'] >= 6).astype(int)
data[['review', 'rating', 'sentiment']].head()

In [ ]:
processed_df, vocab = preprocess_dataframe(
    df=data[['review', 'sentiment']],
    text_col='review',
    label_col='sentiment',
    max_len=30,
    stopwords=DEFAULT_STOPWORDS,
    min_freq=1,
)

dataset = KoreanSentimentDataset(processed_df)
print('vocab size:', len(vocab))
print('dataset size:', len(dataset))
print(processed_df[['review', 'clean_text', 'tokens', 'padded_ids', 'length', 'label']].head())

In [ ]:
val_size = max(1, int(len(dataset) * 0.2)) if len(dataset) > 1 else 0
train_size = len(dataset) - val_size

if val_size > 0:
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
else:
    train_dataset = dataset
    val_dataset = dataset

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = BiLSTMSentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3,
    pad_idx=vocab['<pad>'],
    verbose=False,
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model

In [ ]:
def accuracy_from_probs(probs: torch.Tensor, labels: torch.Tensor) -> float:
    preds = (probs >= 0.5).float()
    return (preds == labels).float().mean().item()

model.train()
for batch_idx, (input_ids, labels, lengths) in enumerate(train_loader):
    input_ids = input_ids.to(device)
    labels = labels.to(device).float().unsqueeze(1)
    lengths = lengths.to(device)

    optimizer.zero_grad(set_to_none=True)
    probs = model(input_ids, lengths)
    loss = criterion(probs, labels)
    loss.backward()
    optimizer.step()

    print('train loss:', float(loss.item()))
    print('train acc :', accuracy_from_probs(probs.detach(), labels.detach()))    

model.eval()
with torch.no_grad():
    for batch_idx, (input_ids, labels, lengths) in enumerate(val_loader):
        input_ids = input_ids.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        lengths = lengths.to(device)
        probs = model(input_ids, lengths)
        print('val acc:', accuracy_from_probs(probs, labels))        